# 2강 정답 노트북 - 생성모형과 우도

이 노트북은 `lesson_02.ipynb`의 기준 구현과 해설을 담은 정답용 자료입니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import binom
from scipy.stats import beta as beta_dist

plt.rcParams["figure.figsize"] = (7, 4)

## 1. 우도 곡선

A 지역과 B 지역의 관측 재취업률은 모두 0.6입니다. 하지만 B 지역은 표본 수가 더 크므로 우도 곡선이 더 좁습니다.

In [ ]:
theta_grid = np.linspace(0.001, 0.999, 500)

lik_A = binom.pmf(12, 20, theta_grid)
lik_B = binom.pmf(120, 200, theta_grid)

rel_lik_A = lik_A / lik_A.max()
rel_lik_B = lik_B / lik_B.max()

plt.plot(theta_grid, rel_lik_A, label="A: n=20, y=12")
plt.plot(theta_grid, rel_lik_B, label="B: n=200, y=120")
plt.xlabel("theta")
plt.ylabel("Relative likelihood")
plt.legend()
plt.show()

theta_A_mle = theta_grid[np.argmax(lik_A)]
theta_B_mle = theta_grid[np.argmax(lik_B)]

print("A likelihood peak near:", round(theta_A_mle, 3))
print("B likelihood peak near:", round(theta_B_mle, 3))
print("sum(lik_A):", lik_A.sum())
print("sum(lik_B):", lik_B.sum())

In [ ]:
assert theta_grid.shape == lik_A.shape == lik_B.shape == rel_lik_A.shape == rel_lik_B.shape
assert abs(theta_A_mle - 0.6) < 0.01
assert abs(theta_B_mle - 0.6) < 0.01
idx_05 = np.argmin(np.abs(theta_grid - 0.5))
assert rel_lik_B[idx_05] < rel_lik_A[idx_05]

print("자동 점검 1 통과")

## 2. 정규화 상수와 주변우도

유한 가설에서는 정규화 상수를 합으로 계산합니다.

\[
P(Y=y)=\sum_\theta P(Y=y\mid\theta)P(\theta)
\]

In [ ]:
theta_candidates = np.array([0.2, 0.5, 0.8])
prior = np.array([0.2, 0.5, 0.3])
n = 5
y = 4

likelihood = binom.pmf(y, n, theta_candidates)
unnormalized = prior * likelihood
evidence = unnormalized.sum()
posterior = unnormalized / evidence

table = pd.DataFrame({
    "theta": theta_candidates,
    "prior": prior,
    "likelihood": likelihood,
    "prior_x_likelihood": unnormalized,
    "posterior": posterior,
})

print("evidence:", evidence)
print("posterior sum:", posterior.sum())
table

In [ ]:
assert np.isclose(prior.sum(), 1)
assert np.isclose(posterior.sum(), 1)
assert evidence > 0
assert posterior[np.argmax(posterior)] == posterior[-1]

print("자동 점검 2 통과")

## 3. 격자 근사

연속형 모수에서는 사후밀도의 합이 아니라 적분값이 1이 되어야 합니다.

In [ ]:
theta_grid_2 = np.linspace(0.001, 0.999, 1000)
alpha_prior = 2
beta_prior = 2
n_obs = 20
y_obs = 12

prior_density = beta_dist(alpha_prior, beta_prior).pdf(theta_grid_2)
likelihood_grid = binom.pmf(y_obs, n_obs, theta_grid_2)
unnormalized_grid = prior_density * likelihood_grid
evidence_grid = np.trapezoid(unnormalized_grid, theta_grid_2)
posterior_density = unnormalized_grid / evidence_grid

posterior_area = np.trapezoid(posterior_density, theta_grid_2)
print("posterior area:", posterior_area)

plt.plot(theta_grid_2, prior_density, label="Prior Beta(2,2)")
plt.plot(theta_grid_2, posterior_density, label="Grid posterior")
plt.xlabel("theta")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
assert theta_grid_2.shape == prior_density.shape == likelihood_grid.shape == posterior_density.shape
assert evidence_grid > 0
assert np.isclose(posterior_area, 1, atol=1e-3)

print("자동 점검 3 통과")

## 4. Beta-Binomial 갱신 함수

이번 함수는 `n, y`를 입력으로 받고, 내부에서 성공 수와 실패 수를 명시합니다.

In [ ]:
def beta_binomial_update(alpha_prior, beta_prior, n, y):
    successes = y
    failures = n - y
    
    alpha_post = alpha_prior + successes
    beta_post = beta_prior + failures
    posterior_mean = alpha_post / (alpha_post + beta_post)
    ci_low, ci_high = beta_dist(alpha_post, beta_post).ppf([0.025, 0.975])
    
    return alpha_post, beta_post, posterior_mean, ci_low, ci_high

test_A = beta_binomial_update(3, 7, 20, 12)
test_B = beta_binomial_update(3, 7, 200, 120)
test_C = beta_binomial_update(2, 2, 5, 3)

print("A:", test_A)
print("B:", test_B)
print("C:", test_C)

In [ ]:
assert test_A[0] == 15 and test_A[1] == 15
assert np.isclose(test_A[2], 0.5)
assert test_B[0] == 123 and test_B[1] == 87
assert np.isclose(test_B[2], 123 / 210)
assert test_C[0] == 5 and test_C[1] == 4
assert test_A[3] < test_A[2] < test_A[4]

print("자동 점검 4 통과")

## 5. 순차 갱신

Beta-Binomial 모형에서는 성공 총수와 실패 총수가 같으면, 한 번에 갱신하든 순차적으로 갱신하든 결과가 같습니다.

In [ ]:
one_step = beta_binomial_update(2, 2, 15, 8)

after_first = beta_binomial_update(2, 2, 5, 3)
after_second = beta_binomial_update(after_first[0], after_first[1], 10, 5)

print("one step:", one_step)
print("after first:", after_first)
print("after second:", after_second)

assert one_step[0] == after_second[0]
assert one_step[1] == after_second[1]
assert np.isclose(one_step[2], after_second[2])

print("자동 점검 5 통과")

# 종합평가 기준 구현

In [ ]:
def grid_posterior(theta_grid, prior_density, n, y):
    likelihood = binom.pmf(y, n, theta_grid)
    unnormalized = prior_density * likelihood
    evidence = np.trapezoid(unnormalized, theta_grid)
    posterior_density = unnormalized / evidence
    return likelihood, posterior_density

eval_grid = np.linspace(0.001, 0.999, 1000)
eval_prior = beta_dist(2, 2).pdf(eval_grid)
eval_likelihood, eval_posterior = grid_posterior(eval_grid, eval_prior, 20, 12)

assert eval_likelihood.shape == eval_grid.shape
assert eval_posterior.shape == eval_grid.shape
assert np.isclose(np.trapezoid(eval_posterior, eval_grid), 1, atol=1e-3)

print("프로그래밍 평가 1 통과")

In [ ]:
def beta_binomial_summary(alpha_prior, beta_prior, n, y):
    successes = y
    failures = n - y
    alpha_post = alpha_prior + successes
    beta_post = beta_prior + failures
    posterior_mean = alpha_post / (alpha_post + beta_post)
    ci_low, ci_high = beta_dist(alpha_post, beta_post).ppf([0.025, 0.975])
    return alpha_post, beta_post, posterior_mean, ci_low, ci_high

summary_1 = beta_binomial_summary(2, 3, 5, 4)
summary_2 = beta_binomial_summary(3, 7, 20, 12)
summary_3 = beta_binomial_summary(1, 1, 10, 0)

assert summary_1[0] == 6 and summary_1[1] == 4
assert np.isclose(summary_1[2], 0.6)
assert summary_2[0] == 15 and summary_2[1] == 15
assert summary_3[0] == 1 and summary_3[1] == 11
assert summary_2[3] < summary_2[2] < summary_2[4]

print("프로그래밍 평가 2 통과")

In [ ]:
local_A = beta_binomial_summary(3, 7, 20, 12)
local_B = beta_binomial_summary(3, 7, 200, 120)

results = pd.DataFrame([
    {
        "local": "A",
        "posterior": f"Beta({local_A[0]}, {local_A[1]})",
        "posterior_mean": local_A[2],
        "95% CI Lower": local_A[3],
        "95% CI Upper": local_A[4],
    },
    {
        "local": "B",
        "posterior": f"Beta({local_B[0]}, {local_B[1]})",
        "posterior_mean": local_B[2],
        "95% CI Lower": local_B[3],
        "95% CI Upper": local_B[4],
    },
])

print(results)

p_grid = np.linspace(0.001, 0.999, 1000)
A_density = beta_dist(local_A[0], local_A[1]).pdf(p_grid)
B_density = beta_dist(local_B[0], local_B[1]).pdf(p_grid)

plt.plot(p_grid, A_density, label="A: Beta(15, 15)")
plt.plot(p_grid, B_density, label="B: Beta(123, 87)")
plt.xlabel("theta")
plt.ylabel("Density")
plt.legend()
plt.show()

print("해석:")
print("두 지역의 관측 재취업률은 모두 0.6입니다.")
print("하지만 B 지역은 표본 수가 200명으로 더 많기 때문에 사전분포의 영향이 상대적으로 작습니다.")
print("따라서 B의 사후평균이 0.6에 더 가깝고, credible interval도 더 좁습니다.")